# Training the Model

In [ ]:
# To start, i'd like to thank Andrej Kaparthy for the tutorial! Pretty easy to follow along!
# https://www.youtube.com/watch?v=kCc8FmEb1nY

import torch
import torch.nn as nn
from torch.nn import functional as F

# parameters
dataset = 'datasets/aurora_dataset.txt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 64
batch_size = 16
training_steps = 100000
eval_interval = 500
eval_iters = 200
learning_rate = 3e-4
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
data_split = 0.9

torch.set_num_threads(24) # Ryzen 9 9900X
torch.manual_seed(6767)

# load dataset
with open(dataset, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
strings_to_ints = { ch:i for i, ch in enumerate(chars) }
ints_to_strings = { i:ch for i, ch in enumerate(chars) }

# endcoder & decoders - encodes / decodes strings to vectors
encode = lambda s: [strings_to_ints[c] for c in s]
decode = lambda l: ''.join([ints_to_strings[i] for i in l])

# split data into training set & validating set
data = torch.tensor(encode(text), dtype=torch.long)
n = int(data_split*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
# functions
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [4]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out
    
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class AuroraGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    """
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
        """
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # apply temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                min_v = v[:, -1].unsqueeze(1)
                logits = torch.where(logits < min_v, torch.full_like(logits, float('-inf')), logits)

            probs = F.softmax(logits, dim=-1)

            if top_p is not None:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                sorted_idx_to_remove = cumulative_probs > top_p
                sorted_idx_to_remove[:, 1:] = sorted_idx_to_remove[:, :-1].clone()
                sorted_idx_to_remove[:, 0] = 0
                mask = torch.zeros_like(probs, dtype=torch.bool).scatter(1, sorted_idx, sorted_idx_to_remove)
                probs = probs.masked_fill(mask, 0.0)
                probs = probs / probs.sum(dim=-1, keepdim=True)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [ ]:
model = AuroraGPT()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(training_steps):
    if iter % eval_interval == 0 or iter == training_steps - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

12.576181 M parameters


'\nfor iter in range(training_steps):\n    if iter % eval_interval == 0 or iter == training_steps - 1:\n        losses = estimate_loss()\n        print(f"step {iter}: train loss {losses[\'train\']:.4f}, val loss {losses[\'val\']:.4f}")\n\n    xb, yb = get_batch(\'train\')\n\n    logits, loss = model(xb, yb)\n    optimizer.zero_grad(set_to_none=True)\n    loss.backward()\n    optimizer.step()\n'

In [ ]:
tokens_to_output = 1000

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(f"output:\n{decode(m.generate(context, max_new_tokens=tokens_to_output)[0].tolist())}")

# Using the model

In [17]:
# save model

import torch
import json
#m.state_dict(), "auroragpt.pt"
torch.save({
    'model_state_dict': m.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "output/auroragpt.pt")

with open("output/aurora_vocab.json", "w") as f:
    json.dump({
        "strings_to_ints": strings_to_ints,
        "ints_to_strings": ints_to_strings
    }, f)

In [18]:
# load model

import torch
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

with open("output/aurora_vocab.json", "r") as f:
    vocab = json.load(f)

strings_to_ints = vocab["strings_to_ints"]
ints_to_strings = vocab["ints_to_strings"]

ints_to_strings = {int(k): v for k, v in ints_to_strings.items()}

# endcoder & decoders - encodes / decodes strings to vectors
encode = lambda s: [strings_to_ints[c] for c in s]
decode = lambda l: ''.join([ints_to_strings[i] for i in l])

model = AuroraGPT().to(device)

m = AuroraGPT().to(device)
pt = torch.load("output/auroragpt.pt", map_location=device)
m.load_state_dict(pt['model_state_dict'])

optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
optimizer.load_state_dict(pt['optimizer_state_dict'])

m.eval()

AuroraGPT(
  (token_embedding_table): Embedding(2485, 384)
  (position_embedding_table): Embedding(64, 384)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedFoward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)


In [23]:
# generating content

prompt = "User: hello,\nAssistant:"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

tokens_to_output = 150
output_tokens = m.generate(
    context,
    max_new_tokens=tokens_to_output,
    temperature=0.85,
    top_k=50,
    top_p=0.9
)

print(decode(output_tokens[0].tolist()))
#context = torch.zeros((1, 1), dtype=torch.long, device=device)
#print(f"output:\n{decode(m.generate(context, max_new_tokens=tokens_to_output)[0].tolist())}")

User: hello,
Assistant: yeah,
User: idk what i saw,
Assistant: you told me the mechanic one,
User: good,
Assistant: I still play on the game but i had the new new command an
